# SegFormer3D (Keras 3 + Torch backend) — TFRecords → `tf.data` → PyTorch loop

This notebook:
- Reads 3D volumes from TFRecords (`.tfrec`) with `tf.data`
- Applies `medicai` 3D augmentations
- Builds a **Keras 3** `medicai.models.SegFormer` model with `KERAS_BACKEND="torch"`
- Trains with a **pure PyTorch-style custom loop** (manual `backward()` + optimizer step)
- Validates with `SlidingWindowInference` on full volumes

H100 tips:
- Keep TensorFlow on **CPU only** so it doesn't reserve GPU VRAM.
- Use BF16 autocast (`torch.autocast(..., dtype=torch.bfloat16)`) for speed + stability.


In [ ]:
# (Kaggle) Optional installs
# If you use an offline wheel bundle:
# var = "/kaggle/input/vsdetection-packages-offline-installer-only/whls"
# !pip install "$var"/keras_nightly-*.whl --no-index --find-links "$var"
# !pip install git+https://github.com/innat/medic-ai.git -q

# If internet is enabled:
# !pip install -U keras -q
# !pip install git+https://github.com/innat/medic-ai.git -q


In [ ]:
import os
import warnings

os.environ.setdefault("KERAS_BACKEND", "torch")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
warnings.filterwarnings("ignore")

import glob
import random

import numpy as np
import tensorflow as tf

# Keep TF on CPU so Torch owns the GPU VRAM.
try:
    tf.config.set_visible_devices([], "GPU")
except Exception:
    pass

import torch
from tqdm.auto import tqdm

import keras
from keras import ops
from keras.optimizers.schedules import CosineDecay

import medicai
from medicai.transforms import (
    Compose,
    NormalizeIntensity,
    RandCutOut,
    RandFlip,
    RandRotate90,
    RandShiftIntensity,
    RandSpatialCrop,
)
from medicai.models import SegFormer
from medicai.losses import SparseCenterlineDiceLoss, SparseDiceCELoss
from medicai.metrics import SparseDiceMetric
from medicai.utils import SlidingWindowInference


In [ ]:
print("keras:", keras.__version__)
print("keras backend:", keras.config.backend())
print("torch:", torch.__version__)
print("tf:", tf.__version__)
print("cuda available:", torch.cuda.is_available())


In [ ]:
# =====================
# Config (H100 defaults)
# =====================

# TFRecord location (edit for your dataset)
TFRECORD_GLOB = "/kaggle/input/vesuvius-tfrecords/*.tfrec"

# Full volume shape stored in TFRecords (used for reshape)
ORIG_SHAPE = (320, 320, 320)

# Train patch size (H100: try 160^3 or 192^3 if it fits)
PATCH_SHAPE = (160, 160, 160)

NUM_CLASSES = 3
IGNORE_LABEL = 2

BATCH_SIZE = 2
EPOCHS = 50
VAL_EVERY = 2

# Gradient accumulation (effective batch = BATCH_SIZE * ACCUM_STEPS)
ACCUM_STEPS = 1

# Mixed precision
USE_BF16 = True  # H100: bf16 is fast + stable

# Optim
GLOBAL_CLIPNORM = 1.0
WEIGHT_DECAY = 1e-5

# Loss
CLDICE_ITERS = 10
CLDICE_WEIGHT = 1.0

# Assumption from the referenced notebook: each .tfrec has 6 samples
SAMPLES_PER_TFREC = 6

# Repro
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)


In [ ]:
# =====================
# TFRecord schema helper
# =====================

def describe_tfrecord(path: str, n: int = 1) -> None:
    """Print TFRecord feature keys/types for the first n records."""
    raw_ds = tf.data.TFRecordDataset([path]).take(n)
    for raw in raw_ds:
        ex = tf.train.Example.FromString(raw.numpy())
        print("Features:", list(ex.features.feature.keys()))
        for k, v in ex.features.feature.items():
            kind = v.WhichOneof("kind")
            if kind == "bytes_list":
                ln = len(v.bytes_list.value[0]) if v.bytes_list.value else 0
                print(f" - {k}: bytes_list (len0={ln})")
            elif kind == "int64_list":
                vals = list(v.int64_list.value)
                print(f" - {k}: int64_list (len={len(vals)}) {vals[:8]}")
            elif kind == "float_list":
                vals = list(v.float_list.value)
                print(f" - {k}: float_list (len={len(vals)}) {vals[:8]}")
            else:
                print(f" - {k}: {kind}")


all_tfrec = sorted(
    glob.glob(TFRECORD_GLOB),
    key=lambda x: int(os.path.splitext(x)[0].split("_")[-1])
    if os.path.splitext(x)[0].split("_")[-1].isdigit()
    else x,
)
print("TFRecords found:", len(all_tfrec))
if all_tfrec:
    describe_tfrecord(all_tfrec[0], n=1)
else:
    print("No TFRecords found. Update TFRECORD_GLOB.")


In [ ]:
# =====================
# TFRecord parsing
# =====================

# Update these keys/dtypes if your TFRecords differ.
IMAGE_KEY = "image"
LABEL_KEY = "label"
SHAPE_KEY = "shape"  # int64[3] = (D,H,W)

IMAGE_DTYPE = tf.uint16
LABEL_DTYPE = tf.uint8

FEATURE_DESC = {
    IMAGE_KEY: tf.io.FixedLenFeature([], tf.string),
    LABEL_KEY: tf.io.FixedLenFeature([], tf.string),
    SHAPE_KEY: tf.io.FixedLenFeature([3], tf.int64, default_value=list(ORIG_SHAPE)),
}


def parse_tfrecord_fn(example_proto: tf.Tensor):
    ex = tf.io.parse_single_example(example_proto, FEATURE_DESC)
    shape = tf.cast(ex[SHAPE_KEY], tf.int32)
    image = tf.io.decode_raw(ex[IMAGE_KEY], out_type=IMAGE_DTYPE)
    label = tf.io.decode_raw(ex[LABEL_KEY], out_type=LABEL_DTYPE)
    image = tf.reshape(image, shape)
    label = tf.reshape(label, shape)
    return image, label


def prepare_inputs(image: tf.Tensor, label: tf.Tensor):
    # Add channel dim: (D,H,W) -> (D,H,W,1)
    image = tf.cast(image, tf.float32)[..., None]
    label = tf.cast(label, tf.int32)[..., None]
    return image, label


In [ ]:
# =====================
# Augmentations (medicai)
# =====================

def train_transformation(image, label):
    data = {"image": image, "label": label}
    pipeline = Compose(
        [
            RandSpatialCrop(
                keys=["image", "label"],
                roi_size=PATCH_SHAPE,
                random_center=True,
                random_size=False,
                invalid_label=IGNORE_LABEL,
                min_valid_ratio=0.5,
                max_attempts=10,
            ),
            RandFlip(keys=["image", "label"], spatial_axis=[0], prob=0.5),
            RandFlip(keys=["image", "label"], spatial_axis=[1], prob=0.5),
            RandFlip(keys=["image", "label"], spatial_axis=[2], prob=0.5),
            RandRotate90(
                keys=["image", "label"],
                prob=0.4,
                max_k=3,
                spatial_axes=(0, 1),
            ),
            NormalizeIntensity(keys=["image"], nonzero=True, channel_wise=False),
            RandShiftIntensity(keys=["image"], offsets=0.10, prob=0.5),
            RandCutOut(
                keys=["image", "label"],
                invalid_label=IGNORE_LABEL,
                mask_size=[PATCH_SHAPE[1] // 4, PATCH_SHAPE[2] // 4],
                fill_mode="constant",
                cutout_mode="volume",
                prob=0.2,
                num_cuts=2,
            ),
        ]
    )
    result = pipeline(data)
    return result["image"], result["label"]


def val_transformation(image, label):
    data = {"image": image, "label": label}
    pipeline = Compose(
        [
            NormalizeIntensity(keys=["image"], nonzero=True, channel_wise=False),
        ]
    )
    result = pipeline(data)
    return result["image"], result["label"]


In [ ]:
# =====================
# tf.data loader
# =====================

def tfrecord_loader(files_or_glob, batch_size: int, shuffle: bool):
    if isinstance(files_or_glob, (list, tuple)):
        files = list(files_or_glob)
    else:
        files = tf.io.gfile.glob(files_or_glob)

    if not files:
        raise FileNotFoundError(f"No TFRecords found for: {files_or_glob}")

    ds = tf.data.TFRecordDataset(files, num_parallel_reads=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(buffer_size=256, reshuffle_each_iteration=True)

    ds = ds.map(parse_tfrecord_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(prepare_inputs, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.map(train_transformation, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(val_transformation, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(batch_size, drop_remainder=shuffle)

    options = tf.data.Options()
    options.experimental_deterministic = False
    ds = ds.with_options(options)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


# Simple hold-out split (for high score: prefer grouped K-fold over volumes/fragments)
val_idx = -1
val_files = [all_tfrec[val_idx]] if all_tfrec else []
train_files = [
    f for i, f in enumerate(all_tfrec) if i != len(all_tfrec) + val_idx
]

if all_tfrec:
    train_ds = tfrecord_loader(train_files, batch_size=BATCH_SIZE, shuffle=True)
    val_ds = tfrecord_loader(val_files, batch_size=1, shuffle=False)

    x, y = next(iter(val_ds))
    print("val batch:", x.shape, y.shape, x.dtype, y.dtype)


In [ ]:
# =====================
# Model (medicai SegFormer3D)
# =====================

model = SegFormer(
    input_shape=PATCH_SHAPE + (1,),
    encoder_name="mit_b2",  # b0 baseline; b2/b3 often better if memory allows
    classifier_activation="softmax",
    num_classes=NUM_CLASSES,
    dropout=0.2,
)

# Build on the target device
dummy = torch.zeros((1, *PATCH_SHAPE, 1), device=DEVICE)
_ = model(dummy, training=False)

print("Params (M):", model.count_params() / 1e6)
try:
    print(model.instance_describe())
except Exception:
    pass


In [ ]:
# =====================
# Optim + loss + metrics
# =====================

num_train_samples = max(1, len(train_files) * SAMPLES_PER_TFREC)
steps_per_epoch = max(1, (num_train_samples // BATCH_SIZE) // ACCUM_STEPS)
total_steps = steps_per_epoch * EPOCHS

warmup_steps = int(total_steps * 0.05)
decay_steps = max(1, total_steps - warmup_steps)

lr_schedule = CosineDecay(
    initial_learning_rate=1e-6,
    decay_steps=decay_steps,
    warmup_target=min(3e-4, 1e-4 * (BATCH_SIZE / 2)),
    warmup_steps=warmup_steps,
    alpha=0.1,
)

optim = keras.optimizers.AdamW(
    learning_rate=lr_schedule,
    weight_decay=WEIGHT_DECAY,
    global_clipnorm=GLOBAL_CLIPNORM,
)

dice_ce_loss_fn = SparseDiceCELoss(
    from_logits=False,
    num_classes=NUM_CLASSES,
    ignore_class_ids=IGNORE_LABEL,
)

cldice_loss_fn = SparseCenterlineDiceLoss(
    from_logits=False,
    num_classes=NUM_CLASSES,
    target_class_ids=1,
    ignore_class_ids=IGNORE_LABEL,
    iters=CLDICE_ITERS,
)


def combined_loss_fn(y_true, y_pred):
    return dice_ce_loss_fn(y_true, y_pred) + (CLDICE_WEIGHT * cldice_loss_fn(y_true, y_pred))


train_metric = SparseDiceMetric(
    from_logits=False,
    num_classes=NUM_CLASSES,
    ignore_class_ids=IGNORE_LABEL,
    name="dice",
)

val_metric = SparseDiceMetric(
    from_logits=False,
    num_classes=NUM_CLASSES,
    ignore_class_ids=IGNORE_LABEL,
    name="val_dice",
)


In [ ]:
# =====================
# Sliding-window inferencer (valid/infer on full volumes)
# =====================

swi = SlidingWindowInference(
    model,
    num_classes=NUM_CLASSES,
    roi_size=PATCH_SHAPE,
    sw_batch_size=2,  # H100: increase if it fits
    overlap=0.5,
    mode="gaussian",
)


In [ ]:
# =====================
# Training loop (PyTorch-style)
# =====================

def _to_torch(x) -> torch.Tensor:
    if isinstance(x, torch.Tensor):
        return x
    if hasattr(x, "numpy"):
        x = x.numpy()
    return torch.from_numpy(x)


def _zero_grads(trainable_weights):
    for w in trainable_weights:
        if w.value.grad is not None:
            w.value.grad = None


trainable_weights = list(model.trainable_weights)
optim.build(trainable_weights)

autocast_enabled = USE_BF16 and (DEVICE.type == "cuda")
autocast_dtype = torch.bfloat16


def train_one_epoch(dataloader):
    model.trainable = True
    train_metric.reset_state()

    running_loss = 0.0
    seen = 0

    loop = tqdm(dataloader, desc="train", leave=False)
    _zero_grads(trainable_weights)

    for step, (imgs, labels) in enumerate(loop, start=1):
        imgs = _to_torch(imgs).to(DEVICE, non_blocking=True)
        labels = _to_torch(labels).to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=autocast_dtype,
            enabled=autocast_enabled,
        ):
            outputs = model(imgs, training=True)
            loss = combined_loss_fn(labels, outputs)
            loss = loss / ACCUM_STEPS

        loss.backward()

        if (step % ACCUM_STEPS) == 0:
            grads = [w.value.grad for w in trainable_weights]
            with torch.no_grad():
                optim.apply(grads, trainable_weights)
            _zero_grads(trainable_weights)

        train_metric.update_state(labels, outputs)

        running_loss += float(ops.convert_to_numpy(loss)) * ACCUM_STEPS
        seen += 1

        loop.set_postfix(
            loss=running_loss / max(1, seen),
            dice=float(ops.convert_to_numpy(train_metric.result())),
        )

    return running_loss / max(1, seen), float(ops.convert_to_numpy(train_metric.result()))


@torch.no_grad()
def validate(dataloader):
    model.trainable = False
    val_metric.reset_state()

    for x, y in tqdm(dataloader, desc="val", leave=False):
        x = _to_torch(x).to(DEVICE, non_blocking=True)
        y = _to_torch(y).to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=autocast_dtype,
            enabled=autocast_enabled,
        ):
            pred = swi(x)
        pred = ops.convert_to_tensor(pred)
        val_metric.update_state(y, pred)

    return float(ops.convert_to_numpy(val_metric.result()))


def run_training(train_loader, val_loader, epochs: int = 10, ckpt_path: str = "segformer3d_best.weights.h5"):
    best = -1.0

    for epoch in range(1, epochs + 1):
        tr_loss, tr_dice = train_one_epoch(train_loader)

        if (epoch % VAL_EVERY) == 0:
            va_dice = validate(val_loader)
            print(f"Epoch {epoch:03d}/{epochs} | loss {tr_loss:.4f} | dice {tr_dice:.4f} | val_dice {va_dice:.4f}")

            if va_dice > best:
                best = va_dice
                model.save_weights(ckpt_path)
                print(f"  ↑ best={best:.4f} saved: {ckpt_path}")
        else:
            print(f"Epoch {epoch:03d}/{epochs} | loss {tr_loss:.4f} | dice {tr_dice:.4f}")

    return best


In [ ]:
# =====================
# Train
# =====================

if all_tfrec:
    best = run_training(train_ds, val_ds, epochs=EPOCHS)
    print("Best val dice:", best)
